In [ ]:

"""
QEvasion Dataset
Install:
pip install datasets pandas scikit-learn pyarrow

This version safely handles:
- NaN values
- empty strings
- Arrow/PyArrow string issues
- unseen labels
- whitespace-only strings
- group-key creation
"""

import re
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from collections import Counter


# loading dataset 
dataset = load_dataset("ailsntua/QEvasion")

train_df = dataset["train"].to_pandas()
test_df  = dataset["test"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)



#step2: Convert Arrow strings → Python strings

train_df = train_df.convert_dtypes(convert_string=False)
test_df  = test_df.convert_dtypes(convert_string=False)

ALL_COLS = set(train_df.columns) | set(test_df.columns)

for col in ALL_COLS:

    if col in train_df.columns:
        train_df[col] = (
            train_df[col]
            .fillna("")
            .astype(str)
            .replace(["None", "<NA>", "nan"], "")
            .str.strip()
        )

    if col in test_df.columns:
        test_df[col] = (
            test_df[col]
            .fillna("")
            .astype(str)
            .replace(["None", "<NA>", "nan"], "")
            .str.strip()
        )



#step3: Drop unnecessary columns


DROP_COLS = [
    "url",
    "annotator1",
    "annotator2",
    "annotator3",
    "gpt3.5_prediction"
]

train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
test_df  = test_df.drop(columns=DROP_COLS, errors="ignore")

print("\nRemaining columns:")
print(train_df.columns.tolist())


Train shape: (3448, 20)
Test shape : (308, 20)

Remaining columns:
['title', 'date', 'president', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'question', 'annotator_id', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']


In [ ]:
# cleam text 
def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"^Q\.\s*", "", text)
    text = re.sub(r"^——\s*", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


TEXT_COLS = [
    "question",
    "interview_question",
    "interview_answer",
    "gpt3.5_summary"
]

for col in TEXT_COLS:

    if col in train_df.columns:
        train_df[col] = train_df[col].apply(clean_text)

    if col in test_df.columns:
        test_df[col] = test_df[col].apply(clean_text)


print("\nSample cleaned question:")
print(train_df["question"].iloc[0])



In [ ]:
# model input 
def build_input(row):

    q = row.get("question", "")
    a = row.get("interview_answer", "")

    return f"[Q]: {q} [A]: {a}"


train_df["model_input"] = train_df.apply(build_input, axis=1)
test_df["model_input"]  = test_df.apply(build_input, axis=1)


print("\nSample model input:")
print(train_df["model_input"].iloc[0][:300])


# clean labels 
LABEL_COLS = ["evasion_label", "clarity_label"]

for col in LABEL_COLS:

    train_df[col] = (
        train_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    test_df[col] = (
        test_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# Remove empty labels ONLY from train
train_df = train_df[
    (train_df["evasion_label"] != "") &
    (train_df["clarity_label"] != "")
]

print("\nTrain rows after label cleaning:", len(train_df))
print("Test rows:", len(test_df))

In [ ]:
# encode evasion labels 

le_evasion = LabelEncoder()

train_df["evasion_label_enc"] = le_evasion.fit_transform(
    train_df["evasion_label"]
)

# map labels 
evasion_mapping = {
    label: idx
    for idx, label in enumerate(le_evasion.classes_)
}

test_df["evasion_label_enc"] = (
    test_df["evasion_label"]
    .map(evasion_mapping)
)

print("\nEvasion label mapping:")
print(evasion_mapping)


# our Binary labels 
REPLY_LABELS = {"Explicit", "Implicit"}

def to_binary(label):

    return 0 if label in REPLY_LABELS else 1


train_df["binary_label"] = train_df["evasion_label"].apply(to_binary)
test_df["binary_label"]  = test_df["evasion_label"].apply(to_binary)


print("\nBinary label distribution:")
print(train_df["binary_label"].value_counts())

In [ ]:
#  encoding clarity labels
le_clarity = LabelEncoder()

train_df["clarity_label_enc"] = le_clarity.fit_transform(
    train_df["clarity_label"]
)

clarity_mapping = {
    label: idx
    for idx, label in enumerate(le_clarity.classes_)
}

test_df["clarity_label_enc"] = (
    test_df["clarity_label"]
    .map(clarity_mapping)
)

print("\nClarity label mapping:")
print(clarity_mapping)


# step 10 converting boolean features into 0,1
BOOL_COLS = [
    "inaudible",
    "multiple_questions",
    "affirmative_questions"
]

for col in BOOL_COLS:

    if col in train_df.columns:
        train_df[col] = (
            train_df[col]
            .replace(["True", "False"], [1, 0])
            .astype(int)
        )

    if col in test_df.columns:
        test_df[col] = (
            test_df[col]
            .replace(["True", "False"], [1, 0])
            .astype(int)
        )


In [ ]:

# president column is encoded here
le_pres = LabelEncoder()

train_df["president_enc"] = le_pres.fit_transform(
    train_df["president"]
)

pres_mapping = {
    label: idx
    for idx, label in enumerate(le_pres.classes_)
}

test_df["president_enc"] = (
    test_df["president"]
    .map(pres_mapping)
)


# converting dates

train_df["date"] = pd.to_datetime(
    train_df["date"],
    errors="coerce"
)

test_df["date"] = pd.to_datetime(
    test_df["date"],
    errors="coerce"
)

train_df["year"] = train_df["date"].dt.year
test_df["year"]  = test_df["date"].dt.year


# creating group key 

for col in ["title", "question_order"]:

    train_df[col] = (
        train_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    test_df[col] = (
        test_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


train_df["group_key"] = (
    train_df["title"]
    + "__"
    + train_df["question_order"]
)

test_df["group_key"] = (
    test_df["title"]
    + "__"
    + test_df["question_order"]
)

print("\nUnique train groups:",
      train_df["group_key"].nunique())


In [ ]:

# train test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        train_df,
        train_df["binary_label"],
        groups=train_df["group_key"]
    )
)

final_train = train_df.iloc[train_idx].reset_index(drop=True)
final_val   = train_df.iloc[val_idx].reset_index(drop=True)

print("\nFinal train:", len(final_train))
print("Final val  :", len(final_val))
print("Test       :", len(test_df))


# here we verify to see that no leakage happens for test dataset
overlap = (
    set(final_train["group_key"]) &
    set(final_val["group_key"])
)

print("\nGroup overlap:", len(overlap))


# ─────────────────────────────────────────────
# STEP 15: Class imbalance
# ─────────────────────────────────────────────

label_counts = Counter(final_train["evasion_label"])

total = len(final_train)

class_weights = {
    label: total / (len(label_counts) * count)
    for label, count in label_counts.items()
}

print("\nClass weights:")

for label, weight in sorted(
    class_weights.items(),
    key=lambda x: -x[1]
):
    print(f"{label:<25} {weight:.3f}")


#step15 out final columns saved for input 

FEATURE_COLS = [
    "model_input",
    "inaudible",
    "multiple_questions",
    "affirmative_questions",
    "president_enc",
    "year",
]

TARGET_COLS = [
    "evasion_label",
    "evasion_label_enc",
    "binary_label",
    "clarity_label",
    "clarity_label_enc",
]

OPTIONAL_COLS = [
    "gpt3.5_summary",
    "group_key",
]

SAVE_COLS = (
    FEATURE_COLS +
    TARGET_COLS +
    OPTIONAL_COLS
)

In [ ]:

#saving files
final_train[SAVE_COLS].to_csv(
    "qevasion_train.csv",
    index=False
)

final_val[SAVE_COLS].to_csv(
    "qevasion_val.csv",
    index=False
)

test_df[SAVE_COLS].to_csv(
    "qevasion_test.csv",
    index=False
)

print("\nSaved files:")
print("qevasion_train.csv")
print("qevasion_val.csv")
print("qevasion_test.csv")

print("\nPreprocessing complete.")